In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import ollama as ol

In [2]:
ol.list()

ListResponse(models=[Model(model='llava:latest', modified_at=datetime.datetime(2026, 10, 7, 19, 30, 11, 946808, tzinfo=TzInfo(-14400)), digest='8dd30f6b0cb19f555f2c7a7ebda861449ea2cc76bf1f44e262931f45fc81d081', size=4733363377, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama', 'clip'], parameter_size='7B', quantization_level='Q4_0')), Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 10, 7, 15, 50, 20, 60485, tzinfo=TzInfo(-14400)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))])

In [3]:
results = pd.read_csv('data/results.csv', index_col=0)
weights = pd.read_csv('data/weights.csv', index_col=0)

In [4]:
#makes the data readable for the LLM
results_text = results.round(3).to_string()
weights_text = (weights['weight'] * 100).round(1).to_string() #makes the weights into a percentage and rounds

In [5]:
SYSTEM_PROMPT = 'an analyst explaining the results'

user_input = f"""
a GA chose one fixed set of weights across 7 funds using 2005-2019, then the weights were held unchanged for 2020-2026

GA portfolio weights in percentages from 2005-2019
{weights_text}

test period results from 2020-2026
{results_text}

Hypothesis: to be hopeful, the hypothesis was that on 2020-2026, the GA portfolio's Sharpe ratio will be higher than the baseline of 100%SPY and equal weight

Questions: 
1. Was the hypothesis supported? Use the numbers.
2. Why do you think the result came out this way?
3. What is one limitation of this experiment?
"""

In [6]:
#run 1
response = ol.chat(
    model='llama3.2',
    messages=[{"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": user_input}],
    options={'temperature': 0, 'seed': 6449},    
)
reply = response["message"]["content"]
print(reply)

**1. Was the hypothesis supported?**

No, the hypothesis was not supported. The GA portfolio's Sharpe ratio (0.558) was lower than the baseline of 100% SPY (0.817) and equal weight (0.646).

**2. Why do you think the result came out this way?**

There are several possible reasons why the GA portfolio's Sharpe ratio was lower than expected:

* The GA portfolio's weights were optimized for the 2005-2019 period, which may not have been representative of the 2020-2026 period. The weights may have been too conservative or too aggressive for the new market conditions.
* The GA portfolio's weights were held unchanged for the 2020-2026 period, which may not have allowed for sufficient adaptation to the changing market conditions. This could have resulted in suboptimal performance.
* The GA portfolio's Sharpe ratio is sensitive to the volatility of the underlying assets. The 2020-2026 period was marked by high volatility, which may have negatively impacted the GA portfolio's Sharpe ratio.

**3.